# 01 · Elicitation notebook (assignment Phase B)

This notebook is Group 9's **bounded domain-information source** for requirements elicitation. It answers questions **only** from the pinned source pack, and it answers by quoting: every answer is the passages that best match the question, word for word, each with a locator (source ID · file · section or page · line). It writes no text of its own, so it cannot claim something no source says. It *can* return passages that do not answer the question, so the team verifies every important answer in `elicitation/notebook-interview.md`.

It is not an oracle and not a substitute for the dataset: the sample database is not part of the pack. The team uses the database to *check* answers, never as the notebook.

| Tier | Sources | Used for |
|---|---|---|
| A, authoritative | A1 to A9: MSR 2027 challenge page, GitSkills and SpecMine preprints, Zenodo records, dataset cards, SpecMine GitHub mirror, GitSkills sample README | Facts about the datasets |
| G, group | G1 to G8: Group 9's research design, data intake, rules and validation, threats, process, project management, Git practice, pipeline API | What the group decided and how it works |
| P, practitioner (optional) | P1 to P5: one member's personal build method | Process questions only |

**Before running:** build the pack once with `python scripts/build_notebook_sources.py` (add `--practice-dir <folder>` for tier P). The reproducible path for the interview is `python -m msr_pipeline elicit`; this notebook calls the same code (`msr_pipeline.elicitation`).

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from msr_pipeline.config import get_paths
from msr_pipeline.elicitation import SCOPES, Notebook, load_questions, render_transcript

ROOT = get_paths().ROOT
PACK = ROOT / "build" / "notebook_sources"
notebook = Notebook.from_pack(PACK)
print("Indexed:", notebook.summary())

m = notebook.manifest
print("Pack built", m["generated_at"], "from repository commit", m["repo_commit"][:12],
      "| tier P included:", m["practice_included"])
pd.DataFrame(
    [{"id": s["id"], "tier": s["tier"], "file": s["file"],
      "version": ", ".join(f"{k} {v}" for k, v in s.get("version", {}).items()),
      "sha256": s.get("sha256", "")[:12]} for s in m["sources"]]
)

## Ask a question

Pick the narrowest scope that fits. A GitSkills question should use `gitskills`, so it is not answered from the SpecMine data dictionary.

| Scope | Searches |
|---|---|
| `gitskills` | A1, A2, A4, A6, A9 |
| `specmine` | A1, A3, A5, A7, A8 |
| `dataset` | every tier-A source |
| `group` | tier G |
| `process` | tiers G and P |
| `all` | everything |

The status line is mechanical: **match** means at least half of the question's terms occur in the returned passages. It does not mean the passages answer the question; that is the team's call. The terms no passage contains are listed, and they often point at what the sources do not establish.

In [ ]:
question = "What does `first_commit_at` represent for a renamed file?"
display(Markdown(notebook.ask(question, scope="gitskills", k=3).to_markdown()))

## Run the prepared interview

The questions live in `elicitation/questions.yaml`. Each is asked word for word, in order, in its stated scope. Follow-ups (`NB-Q05a`, ...) were added after the first pass when an answer was incomplete; they are new questions, never rewordings. Running this cell rewrites `elicitation/notebook-transcript.md`, exactly as `python -m msr_pipeline elicit` does.

In [ ]:
questions = load_questions(ROOT / "elicitation" / "questions.yaml")
answers = notebook.interview(questions, k=3)
(ROOT / "elicitation" / "notebook-transcript.md").write_text(
    render_transcript(answers, notebook, k=3), encoding="utf-8"
)
pd.DataFrame(
    [{"id": a.qid, "kind": a.kind, "scope": a.scope, "status": a.status,
      "terms found": f"{len(a.covered)}/{len(a.terms)}", "sources": ", ".join(a.sources_cited),
      "follow-up of": a.follow_up_of} for a in answers]
)

In [ ]:
# Read any single answer in full.
display(Markdown(answers[0].to_markdown()))

## Verify an answer against the data

Verification is the team's job, not the notebook's. Where a claim is about the data, check it against the sample database with read-only SQL. Example: the documentation says `content_sha_ok` is 1 when the bytes reproduce `file_sha` and 2 when repaired through the blob API, and the preprint says 42 representatives in the full dataset could not be recovered, without naming the value that marks them (NB-Q05, NB-Q05a).

In [ ]:
from msr_pipeline.load import DatasetNotFoundError, query_gitskills

try:
    display(query_gitskills(
        "SELECT content_sha_ok, COUNT(*) AS representatives "
        "FROM artifacts WHERE dedup_primary = 1 GROUP BY content_sha_ok ORDER BY 1"
    ))
except DatasetNotFoundError as exc:
    print("Sample not downloaded; run `make data` first.", exc)